# Lab 02 · วัดผล Retrieval ด้วย Benchmark จริง: BM25 vs Dense vs Hybrid
**2 ต.ค. 09.00–10.00 · ~50 นาที**

## ทำไมต้องมี lab นี้
Lab 01 ได้ hit@1 = 1.00 จาก golden set 8 ข้อ — ฟังดูดี แต่บอกอะไรได้น้อย:
- เอกสารมีแค่ 3 หน้า → เดาถูกง่าย
- 8 ข้อ → พลาด 1 ข้อคะแนนหาย 12.5%
- ไม่มีระบบอื่นให้เทียบ → ไม่รู้ว่า "ดี" เมื่อเทียบกับอะไร

วันนี้ใช้ **benchmark จริง** เพื่อตอบคำถาม: *ค้นแบบจับคำ (BM25) กับค้นแบบความหมาย (bge-m3) แบบไหนดีกว่า สำหรับภาษาไทย — และรวมกันดีกว่าไหม?*

```
                 ┌── A: BM25 (จับคำ)  ──┐
 1,190 คำถาม ──▶ ├── B: Dense (ความหมาย) ─┼──▶ อันดับผลลัพธ์ ──▶ เทียบกับคำตอบที่ถูก (qrels) ──▶ Recall / MRR / nDCG
                 └── C: Hybrid (รวม A+B) ─┘
```

## 1 · ชุดข้อมูล Mr. TyDi (ภาษาไทย)
Benchmark สำหรับ retrieval หลายภาษา สร้างจาก Wikipedia — คำถามเขียนโดยคนจริงที่ **ไม่เห็นคำตอบก่อน** (จึงไม่ลอกคำจากเอกสาร)

| ส่วน | คือ | เทียบกับ Lab 01 |
|---|---|---|
| **queries** | คำถาม 1,190 ข้อ | คำถามใน golden set |
| **qrels** | คำถามข้อนี้ → passage ไหนคือคำตอบ | (ไฟล์, หน้า) ที่ถูก |
| **corpus** | passage ย่อหน้าละชิ้น | chunk |

corpus เต็มมี ~570,000 passages — เราตัดเหลือ **10,000** (`scripts/prepare_tydi.py`) ประกอบด้วย:
- passage ที่เป็นคำตอบ ทุกอัน
- passage อื่นจาก **บทความเดียวกัน** กับคำตอบ → ตัวหลอกที่ยาก (เรื่องเดียวกัน แต่ไม่ใช่คำตอบ)
- passage สุ่ม → ตัวหลอกทั่วไป

In [ ]:
import json
from collections import defaultdict
from cdd_rag.config import ROOT

DATA = ROOT / "data" / "tydi"

queries = dict(line.rstrip("\n").split("\t", 1) for line in open(DATA / "queries.tsv", encoding="utf-8"))
qrels = defaultdict(set)
for line in open(DATA / "qrels.tsv", encoding="utf-8"):
    qid, docid = line.split()
    qrels[qid].add(docid)
corpus = [json.loads(line) for line in open(DATA / "corpus.jsonl", encoding="utf-8")]
doc_by_id = {d["docid"]: d for d in corpus}

print(f"คำถาม {len(queries):,} · คำตอบที่ถูก {sum(map(len, qrels.values())):,} คู่ · corpus {len(corpus):,} passages")

In [ ]:
qid = "4131"
print("❓", queries[qid])
for docid in qrels[qid]:
    d = doc_by_id[docid]
    print(f"✔ [{docid}] {d['title']}: {d['text'][:200]}")

> 💡 ตอนเตรียมข้อมูลพบว่าคำถามบางข้อมี **อักขระล่องหน** (zero-width space `U+200B`) เช่น `สกอตแลนด​์` — ตาเห็นเหมือนปกติ แต่คอมพิวเตอร์ถือเป็นคนละคำ → ลบออกใน `prepare_tydi.py` แล้ว (บทเรียนเดียวกับสระ า ใน Lab 01: **ข้อมูลจริงสกปรกเสมอ**)

---
## 2 · วัดผลอย่างไร: 3 ตัวชี้วัด
แต่ละระบบคืน **อันดับ** passage มา เช่น `[d7, d2, d9, ...]` — เทียบกับคำตอบที่ถูก

| ตัวชี้วัด | ถามว่า | ใช้เมื่อ |
|---|---|---|
| **Recall@k** | คำตอบที่ถูกอยู่ใน k อันดับแรก **ครบแค่ไหน** | RAG ส่ง k chunk ให้ LLM → ขอแค่ "มีอยู่ในนั้น" |
| **MRR@10** | คำตอบถูก **อันแรก** อยู่อันดับที่เท่าไร (1/อันดับ) | ผู้ใช้ดูแค่ผลแรก ๆ |
| **nDCG@10** | ให้คะแนนตามอันดับ — ยิ่งบนยิ่งได้มาก | ตัวชี้วัดมาตรฐานของงาน IR |

(hit@k ใน Lab 01 ≈ Recall@k เมื่อมีคำตอบถูกข้อละ 1 อัน)

In [ ]:
import math

def recall_at_k(ranked, relevant, k):
    return len(set(ranked[:k]) & relevant) / len(relevant)

def mrr_at_k(ranked, relevant, k=10):
    for rank, docid in enumerate(ranked[:k], start=1):
        if docid in relevant:
            return 1 / rank
    return 0.0

def ndcg_at_k(ranked, relevant, k=10):
    dcg = sum(1 / math.log2(rank + 1) for rank, docid in enumerate(ranked[:k], start=1) if docid in relevant)
    ideal = sum(1 / math.log2(rank + 1) for rank in range(1, min(len(relevant), k) + 1))
    return dcg / ideal

# ตัวอย่าง: คำตอบที่ถูกคือ d2 — ระบบ 2 ระบบเรียงต่างกัน
relevant = {"d2"}
for name, ranked in [("ระบบ X", ["d2", "d7", "d9"]), ("ระบบ Y", ["d7", "d9", "d2"])]:
    print(f"{name} {ranked}  Recall@3={recall_at_k(ranked, relevant, 3):.2f}  "
          f"MRR={mrr_at_k(ranked, relevant):.2f}  nDCG={ndcg_at_k(ranked, relevant):.2f}")

**👀 สังเกต:** ทั้งคู่มีคำตอบใน 3 อันดับแรก → Recall@3 เท่ากัน แต่ MRR / nDCG ลงโทษระบบ Y ที่วางคำตอบไว้ล่าง

**🤔 เดาก่อนรัน:** สำหรับ RAG ที่ส่ง 5 chunk ให้ LLM ตัวชี้วัดไหนสำคัญที่สุด?

In [ ]:
def evaluate(run, name):
    """run = {qid: [docid อันดับ 1, 2, ...]} → ค่าเฉลี่ยของทุกคำถาม"""
    scores = {"Recall@5": [], "Recall@10": [], "MRR@10": [], "nDCG@10": []}
    for qid, relevant in qrels.items():
        ranked = run.get(qid, [])
        scores["Recall@5"].append(recall_at_k(ranked, relevant, 5))
        scores["Recall@10"].append(recall_at_k(ranked, relevant, 10))
        scores["MRR@10"].append(mrr_at_k(ranked, relevant))
        scores["nDCG@10"].append(ndcg_at_k(ranked, relevant))
    return {"ระบบ": name, **{m: round(sum(v) / len(v), 4) for m, v in scores.items()}}

results = []

---
## 3 · ระบบ A: BM25 (ค้นแบบจับคำ)
**💡 แนวคิด:** ให้คะแนน passage ตาม **คำที่ตรงกับคำถาม** — คำที่หายาก (เช่น "อยุธยา") มีน้ำหนักมากกว่าคำทั่วไป (เช่น "มี")
เป็นหลักการเดียวกับ TF-IDF แต่ปรับให้ดีขึ้น — ใช้กันมา 30 ปีและยังเป็น baseline ที่เอาชนะยาก

**ปัญหาภาษาไทย:** ไม่เว้นวรรคระหว่างคำ → ต้อง **ตัดคำ** ก่อน (ใช้ pythainlp)

In [ ]:
from pythainlp import word_tokenize

def tokenize(text):
    return [w for w in word_tokenize(text.lower(), engine="newmm", keep_whitespace=False) if w.strip()]

print(tokenize("อยุธยามีกี่อำเภอ ?"))
print(tokenize("โรคเมลิออยด์"))     # ← ชื่อเฉพาะที่ตัวตัดคำไม่รู้จัก ถูกหั่นผิด

In [ ]:
import time
import bm25s

def passage_text(d):
    return f"{d['title']}\n{d['text']}"     # ใส่ชื่อบทความด้วย (เหมือน section ใน Lab 01)

t = time.time()
bm25 = bm25s.BM25()
bm25.index([tokenize(passage_text(d)) for d in corpus], show_progress=False)
print(f"index {len(corpus):,} passages ใช้เวลา {time.time() - t:.1f} วินาที")

In [ ]:
qids = list(qrels)
t = time.time()
idx, _ = bm25.retrieve([tokenize(queries[q]) for q in qids], k=100, show_progress=False)
run_bm25 = {q: [corpus[i]["docid"] for i in row] for q, row in zip(qids, idx)}
print(f"ค้น {len(qids):,} คำถาม ใช้เวลา {time.time() - t:.1f} วินาที")

results.append(evaluate(run_bm25, "A · BM25 + pythainlp"))
results[-1]

---
## 4 · ระบบ B: Dense (bge-m3 + Qdrant)
**💡 แนวคิด:** เหมือน Lab 01 — แปลงเป็นเวกเตอร์แล้วหาที่ใกล้ที่สุด

corpus 10,000 passages ผู้สอน embed ไว้แล้ว และเก็บเป็น snapshot ใน `data/tydi/`
ก่อนเริ่ม lab รัน `uv run python scripts/index_tydi.py` ครั้งเดียว เพื่อโหลด collection `tydi_th_bge_m3` เข้า Qdrant ของเรา (ไม่กี่วินาที ไม่ต้อง embed ใหม่ ถ้าทุกคน embed พร้อมกัน server ไม่ไหว)

In [ ]:
from qdrant_client import models
from cdd_rag.store import client as qdrant
from cdd_rag.embed import embed

TYDI = "tydi_th_bge_m3"
print(TYDI, "มี", qdrant.count(TYDI).count, "points")

In [ ]:
t = time.time()
q_vecs = embed([queries[q] for q in qids])       # คำถามต้องใช้โมเดลเดียวกับ corpus

run_dense = {}
for i in range(0, len(qids), 64):                # ส่งทีละ 64 คำถามในคำขอเดียว
    batch = qids[i : i + 64]
    responses = qdrant.query_batch_points(
        TYDI,
        requests=[models.QueryRequest(query=v, using="dense", limit=100, with_payload=["docid"])
                  for v in q_vecs[i : i + 64]],
    )
    for q, res in zip(batch, responses):
        run_dense[q] = [p.payload["docid"] for p in res.points]
print(f"embed + ค้น {len(qids):,} คำถาม ใช้เวลา {time.time() - t:.1f} วินาที")

results.append(evaluate(run_dense, "B · Dense bge-m3"))
results[-1]

---
## 5 · ระบบ C: Hybrid (รวม A + B)
**💡 สมมติฐาน:** BM25 เก่งเรื่อง **คำเฉพาะ** (ชื่อ ตัวเลข เลขระเบียบ) · Dense เก่งเรื่อง **ความหมาย** → รวมกันน่าจะได้ข้อดีทั้งสอง

**Reciprocal Rank Fusion (RRF):** ไม่สนคะแนนดิบ (สองระบบคะแนนคนละสเกล) — ใช้แค่ **อันดับ**
`score(d) = Σ น้ำหนัก / (60 + อันดับของ d ในแต่ละระบบ)`

In [ ]:
def rrf(weighted_runs, k=60, top=100):
    # weighted_runs = [(run, น้ำหนัก), ...]
    fused = {}
    for q in qids:
        scores = defaultdict(float)
        for run, weight in weighted_runs:
            for rank, docid in enumerate(run.get(q, []), start=1):
                scores[docid] += weight / (k + rank)
        fused[q] = sorted(scores, key=scores.get, reverse=True)[:top]
    return fused

run_hybrid = rrf([(run_bm25, 1.0), (run_dense, 1.0)])    # น้ำหนักเท่ากัน
results.append(evaluate(run_hybrid, "C · Hybrid (RRF 1:1)"))
results[-1]

---
## 6 · เปรียบเทียบ
**🤔 เดาก่อนรัน:** เรียงอันดับ A, B, C จากดีที่สุดไปแย่ที่สุด

In [ ]:
import pandas as pd
pd.DataFrame(results).set_index("ระบบ")

> ⚠️ ตัวเลขสูงกว่าที่รายงานในงานวิจัย เพราะเราใช้ corpus 10,000 passages (ของเต็ม ~570,000) — ใช้ **เทียบระหว่างระบบ** ไม่ใช่ค่าสัมบูรณ์

**👀 ผลจริง (ตอนเตรียม lab):** Dense ชนะขาด (MRR@10 ≈ 0.92 vs BM25 ≈ 0.66) และ **Hybrid แย่กว่า Dense อย่างเดียว** (≈ 0.80) — สมมติฐานผิด!

ทำไม? ไปดูทีละเรื่อง

### 6.1 ทำไม BM25 ภาษาไทยแพ้: การตัดคำ
BM25 เทียบ **คำต่อคำ** — ถ้าตัดคำในคำถามกับในเอกสารไม่ตรงกัน ก็ไม่นับว่าเจอ

In [ ]:
for text in ["อยุธยามีกี่อำเภอ", "จังหวัดพระนครศรีอยุธยาประกอบด้วย 16 อำเภอ",
             "นักพรด คืออะไร", "นักพรต คือผู้บำเพ็ญพรต"]:
    print(f"{text:42} → {tokenize(text)}")

**👀 สังเกต:**
- คำถามใช้ "อยุธยา" แต่เอกสารเป็น "พระนครศรีอยุธยา" ซึ่งตัวตัดคำเก็บเป็น **คำเดียว** → BM25 มองว่าเป็นคนละคำ
- พิมพ์ผิด "นักพรด" ถูกหั่นเป็น `นัก / พร / ด` → ไม่ตรงกับ "นักพรต" เลย · Dense ยังเข้าใจความหมายได้

→ ภาษาที่ไม่เว้นวรรคอย่างไทย ความแม่นของ BM25 **ขึ้นกับตัวตัดคำ** มาก

In [ ]:
def first_rank(run, q):
    return next((r for r, d in enumerate(run[q], start=1) if d in qrels[q]), 999)

dense_only = [q for q in qids if first_rank(run_dense, q) == 1 and first_rank(run_bm25, q) > 10]
print(f"Dense ได้อันดับ 1 แต่ BM25 ไม่ติด top-10: {len(dense_only)} ข้อ — ตัวอย่าง:")
for q in dense_only[:5]:
    ans = doc_by_id[next(iter(qrels[q]))]
    print(f"  ❓ {queries[q]:45}  ✔ {ans['title']}")

### 6.2 แล้ว BM25 มีประโยชน์ไหม?

In [ ]:
bm25_better = [q for q in qids if first_rank(run_bm25, q) < first_rank(run_dense, q)]
dense_better = [q for q in qids if first_rank(run_dense, q) < first_rank(run_bm25, q)]
print(f"BM25 เรียงคำตอบได้ดีกว่า : {len(bm25_better)} ข้อ")
print(f"Dense เรียงได้ดีกว่า    : {len(dense_better)} ข้อ")
print(f"เท่ากัน                 : {len(qids) - len(bm25_better) - len(dense_better)} ข้อ\n")

print("ตัวอย่างที่ BM25 ดีกว่า:")
for q in sorted(bm25_better, key=lambda q: first_rank(run_dense, q) - first_rank(run_bm25, q), reverse=True)[:5]:
    print(f"  BM25 #{first_rank(run_bm25, q):<3} Dense #{first_rank(run_dense, q):<3} ❓ {queries[q]}")

**👀 สังเกต:** BM25 ชนะในคำถามที่มี **ชื่อเฉพาะ** (ชื่อคน ชื่อรุ่นรถ) ที่ตรงกับเอกสารตัวต่อตัว — แต่จำนวนข้อน้อยกว่าที่ Dense ชนะมาก

### 6.3 ปรับน้ำหนัก Hybrid
BM25 อ่อนกว่ามาก → ให้น้ำหนักเท่ากันเท่ากับ "ดึงอันดับดี ๆ ของ Dense ลงมา" · ลองลดน้ำหนัก BM25

In [ ]:
rows = [evaluate(rrf([(run_bm25, w), (run_dense, 1.0)]), f"Hybrid BM25×{w}") for w in (1.0, 0.5, 0.3, 0.1)]
rows.append(evaluate(run_dense, "Dense อย่างเดียว"))
pd.DataFrame(rows).set_index("ระบบ")

**👀 สังเกต:** ลดน้ำหนักแล้วดีขึ้น แต่ **ยังไม่ชนะ Dense อย่างเดียว** บนชุดข้อมูลนี้

### 6.4 ข้อควรระวังก่อนสรุป
- **bge-m3 ถูกฝึกด้วยข้อมูล train ของ Mr. TyDi / MIRACL** (คนละชุดกับ test ที่เราใช้ แต่เป็นคำถามลักษณะเดียวกันจาก Wikipedia) → Dense ได้เปรียบบน benchmark นี้
- เอกสารของกรมฯ ต่างจาก Wikipedia: มี **เลขระเบียบ, ชื่อโครงการ, ตัวย่อ, ศัพท์เฉพาะหน่วยงาน** ที่ bge-m3 อาจไม่เคยเห็น → BM25/hybrid **อาจ** มีประโยชน์มากกว่าที่เห็นที่นี่

**เทียบกับ Lab 01 (เอกสารกำหนดการของเรา, 11 คำถาม):**

| ชุดข้อมูล | dense | sparse/BM25 | hybrid |
|---|---|---|---|
| Lab 01 · เอกสารกรมฯ (มี "ห้อง 303", ชื่อเฉพาะ) | 0.91 | 0.82 | **1.00** |
| Lab 02 · Mr. TyDi (Wikipedia, MRR@10) | **0.92** | 0.66 | 0.80 |

ข้อมูลต่างกัน → วิธีที่ดีที่สุดต่างกัน

**✅ บทเรียนสำคัญที่สุดของ lab นี้:** "hybrid ดีกว่าเสมอ" เป็นความเชื่อ ไม่ใช่ข้อเท็จจริง → **ตัดสินด้วยการวัดบนข้อมูลของเราเอง**

**❓ เช็กความเข้าใจ:** เอกสารของกรมฯ มีเลขระเบียบ ("ระเบียบกรมการพัฒนาชุมชน ว่าด้วย ... พ.ศ. 2566") เยอะ — จะออกแบบการทดลองอย่างไรเพื่อตัดสินว่าควรใช้ hybrid หรือไม่?

> ใน `src/cdd_rag/store.py` ค่าเริ่มต้นเป็น `search(mode="hybrid")` เพราะเอกสารของกรมฯ มีตัวเลข/ชื่อเฉพาะเยอะ (ผล Lab 01) — เปลี่ยนเป็น `"dense"` ได้ถ้าวัดแล้วดีกว่า

---
## 7 · Precision, Recall และ F1 ของการค้น
ตัวชี้วัดในข้อ 2 บอกว่าคำตอบที่ถูกอยู่อันดับไหน อีกมุมหนึ่งที่ใช้กันมากคือ precision กับ recall ซึ่งตอบคำถามคนละแบบ

- precision@k ใน k ชิ้นที่ค้นมาได้ มีกี่ชิ้นที่เกี่ยวข้องจริง ถ้าต่ำแปลว่าเราส่งของที่ไม่เกี่ยวให้ LLM อ่านเยอะ
- recall@k ชิ้นที่เกี่ยวข้องทั้งหมด เราค้นมาได้กี่ชิ้น ถ้าต่ำแปลว่าข้อมูลที่ต้องใช้ตอบตกหล่นไป
- F1 คือค่าเฉลี่ยแบบ harmonic ของสองค่านี้ `2PR / (P + R)` จะสูงได้ก็ต่อเมื่อทั้งสองค่าสูงพร้อมกัน

สำหรับ RAG สองค่านี้ดึงกันคนละทาง ถ้าส่ง chunk ให้ LLM มากขึ้น (k ใหญ่ขึ้น) recall จะขึ้น แต่ precision จะลง เพราะมีของที่ไม่เกี่ยวปนมามากขึ้น

In [ ]:
def prf_at_k(ranked, relevant, k):
    hit = len(set(ranked[:k]) & relevant)            # ใน k ชิ้นแรก มีชิ้นที่ถูกกี่ชิ้น
    p = hit / k
    r = hit / len(relevant)
    f1 = 2 * p * r / (p + r) if hit else 0.0
    return p, r, f1

rows = []
for name, run in [("BM25", run_bm25), ("Dense", run_dense), ("Hybrid", run_hybrid)]:
    for k in (1, 3, 5, 10):
        scores = [prf_at_k(run[q], qrels[q], k) for q in qids]
        p, r, f1 = (sum(s[i] for s in scores) / len(scores) for i in range(3))
        rows.append({"ระบบ": name, "k": k, "Precision": round(p, 3), "Recall": round(r, 3), "F1": round(f1, 3)})

pd.DataFrame(rows).pivot(index="k", columns="ระบบ", values=["Precision", "Recall", "F1"])

ลองดูแนวโน้มเมื่อ k เพิ่มขึ้น recall ขึ้นเกือบเต็ม แต่ precision ลดลงเร็วมาก

ใน Mr. TyDi คำถามส่วนใหญ่มีคำตอบที่ถูกแค่ชิ้นเดียว precision@5 จึงสูงสุดได้ราว 0.2 เท่านั้น แม้ระบบจะหาเจอทุกข้อ ตัวเลข precision จึงต้องอ่านคู่กับจำนวนคำตอบที่ถูกจริงในชุดข้อมูลเสมอ

ค่านี้ใช้ช่วยเลือก `top_k` ของระบบเราได้ ถ้า recall@3 กับ recall@5 ต่างกันน้อย การส่งแค่ 3 ชิ้นให้ LLM จะได้ข้อมูลเกือบเท่าเดิม แต่ LLM อ่านของที่ไม่เกี่ยวน้อยลง ตอบเร็วขึ้น และมีโอกาสถูกเนื้อหาที่ไม่เกี่ยวพาไปผิดทางน้อยลง

---
## 8 · วัดคำตอบของ RAG ทั้งระบบ
ทุกอย่างที่วัดมาถึงตรงนี้วัดแค่ "การค้น" ยังไม่รู้เลยว่าคำตอบสุดท้ายที่ LLM ตอบออกมาถูกหรือไม่ ส่วนนี้จะวัดคำตอบจริง โดยใช้เอกสารกำหนดการและ collection ของเราจาก Lab 01 กับ LLM ตัวกลาง

วิธีที่ง่ายที่สุดคือ token-level F1 (แบบเดียวกับที่ใช้ใน benchmark SQuAD) เขียนคำตอบที่ถูกไว้ล่วงหน้า แล้วตัดคำทั้งคำตอบของระบบและคำตอบที่ถูก จากนั้นนับคำที่ตรงกัน

- precision = คำที่ตรง ÷ จำนวนคำในคำตอบของระบบ ถ้าระบบตอบยืดยาวเกินจำเป็น ค่านี้จะต่ำ
- recall = คำที่ตรง ÷ จำนวนคำในคำตอบที่ถูก ถ้าระบบตอบไม่ครบ ค่านี้จะต่ำ
- F1 รวมสองค่าเข้าด้วยกัน

มีคำถามหนึ่งข้อที่ไม่มีคำตอบในเอกสาร ใช้ดูว่าระบบยอมตอบว่าไม่พบ หรือแต่งคำตอบขึ้นมาเอง

In [ ]:
from cdd_rag.config import settings
from cdd_rag.store import client, ingest_file

# ใช้กำหนดการจาก Lab 01 ถ้า index ไว้แล้วจะข้าม ถ้ายังไม่มี (หรือ collection ว่าง) จะ ingest ให้ใหม่
print(ingest_file(settings.data_dir / "กำหนดการโครงการ.pdf"))
print(settings.collection, "มี", client.count(settings.collection).count, "chunks")

golden_qa = [
    # (คำถาม,                                    คำตอบที่ถูก — ตรวจกับ PDF กำหนดการแล้ว)
    ("การอบรมเรื่อง Docker จัดวันไหน เวลาเท่าไร",     "วันที่ 30 กันยายน 2569 เวลา 15.30 - 17.00 น."),
    ("ใครเป็นวิทยากรเรื่อง Docker",                 "วิทยากรจากศูนย์เทคโนโลยีอิเล็กทรอนิกส์และคอมพิวเตอร์แห่งชาติ"),
    ("Design Thinking สอนวันไหน เวลาเท่าไร",        "วันที่ 28 กันยายน 2569 เวลา 10.00 - 12.00 น."),
    ("หัวข้อ Human-in-the-loop สอนวันไหน",           "วันที่ 1 ตุลาคม 2569 เวลา 15.30 - 17.00 น."),
    ("พักรับประทานอาหารว่างกี่โมง",                  "เวลา 10.30 น. และ 14.30 น."),
    ("ปิดโครงการกี่โมง",                             "17.00 - 17.30 น."),
    ("การอบรมจัดที่ไหน",                             "ห้องอบรม 303 ชั้น 3 อาคารซอฟต์แวร์พาร์ค ถนนแจ้งวัฒนะ"),
    ("ค่าเบี้ยเลี้ยงวิทยากรวันละเท่าไหร่",               "ไม่พบข้อมูลในเอกสาร"),   # ไม่มีในเอกสาร
]

In [ ]:
import re
from collections import Counter

def token_prf(pred, ref):
    """precision / recall / F1 ระดับคำ (ตัดคำไทยด้วย tokenize จากข้อ 3)"""
    pred = re.sub(r"\[\d+\]", " ", pred)            # ตัดเลขอ้างอิง [1] [2] ออกก่อนเทียบ
    p_tok, r_tok = Counter(tokenize(pred)), Counter(tokenize(ref))
    same = sum((p_tok & r_tok).values())             # จำนวนคำที่ตรงกัน (นับซ้ำได้ตามจำนวนที่มีทั้งสองฝั่ง)
    if same == 0:
        return 0.0, 0.0, 0.0
    p, r = same / sum(p_tok.values()), same / sum(r_tok.values())
    return p, r, 2 * p * r / (p + r)

# ลองกับตัวอย่างก่อน
ref = "วันที่ 30 กันยายน 2569 เวลา 15.30 - 17.00 น."
for pred in ["วันที่ 30 กันยายน 2569 เวลา 15.30 - 17.00 น.",
             "30 กันยายน 2569",
             "การอบรมเรื่อง Docker จัดขึ้นในวันที่ 30 กันยายน 2569 ช่วงบ่ายเวลา 15.30 - 17.00 น. โดยวิทยากรจาก NECTEC [1]"]:
    p, r, f1 = token_prf(pred, ref)
    print(f"P={p:.2f} R={r:.2f} F1={f1:.2f} | {pred}")

คำตอบที่สองสั้นแต่ถูก precision จึงเต็ม แต่ recall ต่ำเพราะขาดเวลา ส่วนคำตอบที่สามครบแต่พูดเกินมามาก recall เต็ม แต่ precision ลดลง

ต่อไปให้ RAG ตอบคำถามจริงทุกข้อแล้ววัด

In [ ]:
from cdd_rag.rag import answer

records = []
for question, ref in golden_qa:
    try:
        pred = answer(question)["answer"]
    except Exception as e:
        print("เรียก LLM ไม่ได้:", type(e).__name__, "— ตรวจ LLM_BASE_URL / LLM_MODEL ใน .env")
        break
    p, r, f1 = token_prf(pred, ref)
    records.append({"คำถาม": question, "P": round(p, 2), "R": round(r, 2), "F1": round(f1, 2), "คำตอบของระบบ": pred.replace("\n", " ")[:120]})

qa = pd.DataFrame(records)
if len(qa):
    print(f"ค่าเฉลี่ย  Precision {qa.P.mean():.2f}  Recall {qa.R.mean():.2f}  F1 {qa.F1.mean():.2f}")
    no_answer = qa[qa["คำถาม"].str.contains("เบี้ยเลี้ยง")]
    print("คำถามที่ไม่มีในเอกสาร ระบบตอบว่าไม่พบ:", bool(no_answer["คำตอบของระบบ"].str.contains("ไม่พบ").all()))
qa

อ่านผลรายข้อควบคู่กับค่าเฉลี่ยเสมอ ข้อที่ F1 ต่ำอาจไม่ได้ผิดจริง เพราะ token F1 มีข้อจำกัดชัดเจน

- เขียนต่างแต่ความหมายเดียวกันก็ถูกหักคะแนน เช่น "30 ก.ย." กับ "30 กันยายน" หรือ "บ่ายสามโมงครึ่ง" กับ "15.30"
- คำตอบที่ยาวและให้รายละเอียดดีจะได้ precision ต่ำ ทั้งที่ผู้ใช้อาจชอบ
- ไม่ได้ตรวจว่าข้อมูลมาจากเอกสารจริงหรือ LLM แต่งขึ้นเอง

ในงานจริงจึงมักใช้คู่กับวิธีอื่น เช่น ให้ LLM อีกตัวเป็นผู้ตรวจ (LLM-as-judge) ว่าคำตอบถูกต้องและอิงจากเอกสารหรือไม่ หรือให้เจ้าหน้าที่สุ่มตรวจ token F1 เหมาะกับการเทียบก่อนและหลังปรับระบบ เพราะคำนวณเร็ว ได้ผลเหมือนเดิมทุกครั้ง และไม่ต้องเสียค่า LLM

ลองทำเอง เปลี่ยน `top_k` ใน `answer(question, top_k=3)` แล้วดูว่า F1 เปลี่ยนไปอย่างไร เทียบกับผล recall@k ในข้อ 7

---
## 9 · ขั้นที่ 2: วัดบนเอกสารจริงของกรมฯ
ถึงตรงนี้เราวัดกันบน Mr. TyDi ซึ่งเป็น base eval คือใช้ตรวจว่าชิ้นส่วนของระบบ (โมเดล วิธีค้น การตัดคำ) ใช้กับภาษาไทยได้หรือไม่ ถ้าไม่ผ่านขั้นนี้ก็ไม่ต้องไปต่อ

แต่ Mr. TyDi บอกไม่ได้ว่าระบบพร้อมใช้กับงานของหน่วยงานหรือยัง เพราะเนื้อหาต่างกันมาก Wikipedia ไม่มีเลขมาตรา ชื่อโครงการ หรือตัวย่ออย่าง กข.คจ. ขั้นที่ 2 จึงต้องวัดบนเอกสารจริงของกรมฯ

| | ขั้นที่ 1 · Mr. TyDi | ขั้นที่ 2 · เอกสารกรมฯ |
|---|---|---|
| ใช้ตอบคำถาม | ชิ้นส่วนของระบบใช้กับภาษาไทยได้ไหม | ระบบพร้อมใช้กับงานของเราหรือยัง |
| ใครสร้างชุดทดสอบ | นักวิจัย | เจ้าหน้าที่ที่รู้เนื้อหา |
| ขนาด | 1,190 คำถาม | ในตัวอย่างนี้ 27 คำถาม (งานจริงควรมี 50 ขึ้นไป) |

เอกสารที่ใช้เป็นเอกสารเผยแพร่ของกรมฯ ได้แก่ คู่มือปฏิบัติงานด้านกฎหมายเกี่ยวกับกองทุน คู่มือกฎหมายกองทุนหมู่บ้าน (60 หน้าแรก) สรุปบทบาทและภารกิจกรมฯ และเอกสารเผยแพร่เรื่องทุนชุมชน รวมกับกำหนดการจาก Lab 01

ชุดทดสอบอยู่ที่ `data/golden/cdd_testset.csv` แบ่งคำถามเป็น 5 แบบ

- ตรง ใช้คำใกล้เคียงกับเอกสาร
- ตัวเลข ถามจำนวน วันที่ หรือระยะเวลา
- อ้อม ใช้คำต่างจากเอกสาร เช่น ถามถึง "เจ้าหน้าที่ที่ลงพื้นที่" แทนคำว่า "พัฒนากร"
- ตัวหลอก คู่มือกองทุนมีสองส่วนที่หัวข้อเหมือนกัน คือ กข.คจ. กับกองทุนพัฒนาบทบาทสตรี เช่น "ใครเป็นผู้ลงนามในสัญญา" มีทั้งสองส่วน ระบบต้องเลือกส่วนให้ถูก
- ไม่มีคำตอบ ไม่มีในเอกสารเลย ระบบควรตอบว่าไม่พบ

หมายเหตุ ชุดทดสอบนี้ร่างขึ้นสำหรับการสอน และตรวจแล้วว่าทุกข้อมีหลักฐานอยู่ในหน้าที่ระบุจริง แต่ถ้าจะใช้กับงานจริง ต้องให้เจ้าหน้าที่ที่รู้เนื้อหาตรวจและเพิ่มคำถามอีก

In [ ]:
import csv
from cdd_rag.store import ingest_dir, search

# เพิ่มเอกสารใหม่เข้า collection ของเรา (ไฟล์ที่ index แล้วจะข้าม) ใช้เวลาประมาณ 1 นาที
print(ingest_dir())

testset = list(csv.DictReader(open(ROOT / "data/golden/cdd_testset.csv", encoding="utf-8")))
answerable = [r for r in testset if r["file_name"]]
print(f"\nคำถามทั้งหมด {len(testset)} ข้อ มีคำตอบในเอกสาร {len(answerable)} ข้อ")
print(pd.Series([r["type"] for r in testset]).value_counts().to_dict())

ใช้ตัวชี้วัดชุดเดียวกับ Mr. TyDi ถ้าระบบดึงหน้าที่ถูกมาได้ ถือว่าค้นถูก บางข้อมีคำตอบมากกว่าหนึ่งหน้า จึงระบุไว้หลายหน้า เช่น `9|13`

In [ ]:
def gold_pages(r):
    return {(r["file_name"], int(p)) for p in r["pages"].split("|")}

def eval_cdd(mode, k=5):
    scores, first_hit = defaultdict(list), {}
    for r in answerable:
        ranked = [(h["file_name"], h["page"]) for h in search(r["question"], top_k=10, mode=mode)]
        gold = gold_pages(r)
        first = next((i for i, x in enumerate(ranked, start=1) if x in gold), None)
        first_hit[r["id"]] = first
        tp = sum(x in gold for x in ranked[:k])
        p, rc = tp / k, len(gold & set(ranked[:k])) / len(gold)
        scores["hit@1"].append(first == 1)
        scores["hit@5"].append(first is not None and first <= 5)
        scores["MRR@10"].append(1 / first if first else 0.0)
        scores[f"P@{k}"].append(p)
        scores[f"R@{k}"].append(rc)
        scores[f"F1@{k}"].append(2 * p * rc / (p + rc) if p + rc else 0.0)
    return {m: round(sum(v) / len(v), 3) for m, v in scores.items()}, first_hit

cdd_results, cdd_first = {}, {}
for mode in ("dense", "sparse", "hybrid"):
    cdd_results[mode], cdd_first[mode] = eval_cdd(mode)
pd.DataFrame(cdd_results).T

ผลที่วัดไว้ตอนเตรียม lab (bge-m3 ตัวเดียวกัน)

| | dense | sparse | hybrid |
|---|---|---|---|
| hit@1 | 0.83 | 0.65 | 0.87 |
| hit@5 | 1.00 | 0.91 | 1.00 |
| MRR@10 | 0.91 | 0.77 | 0.92 |
| F1@5 | 0.53 | 0.39 | 0.50 |

เทียบกับขั้นที่ 1 จะเห็นว่าผลไม่เหมือนกัน บน Mr. TyDi แบบ dense ชนะทุกตัวชี้วัด และ hybrid แย่กว่า dense แต่บนเอกสารของกรมฯ hybrid ได้ hit@1 และ MRR สูงที่สุด เพราะคำถามมีชื่อโครงการและตัวย่ออย่าง กข.คจ. ที่การค้นแบบจับคำช่วยได้ ส่วน F1@5 ของ dense ยังสูงกว่าเล็กน้อย

นี่คือเหตุผลที่ต้องวัดทั้งสองขั้น benchmark ใช้เลือกชิ้นส่วน แต่จะตั้งค่าแบบไหนให้ดูจากข้อมูลของเราเอง

ต่อไปดูทีละข้อ ตัวเลขคืออันดับแรกที่ค้นเจอหน้าที่ถูก ถ้าเป็น `-` แปลว่าไม่ติด 10 อันดับแรก

In [ ]:
per_q = pd.DataFrame([
    {"id": r["id"], "ประเภท": r["type"], "คำถาม": r["question"][:45],
     **{m: cdd_first[m][r["id"]] or "-" for m in cdd_first}}
    for r in answerable
]).set_index("id")
per_q

ลองหาข้อที่ยังพลาดหรือได้อันดับต่ำ แล้วคิดว่าเกิดจากอะไร จากตอนเตรียม lab มีข้อสังเกตดังนี้

- ข้อที่ถามอ้อมมาก เช่น "เจ้าหน้าที่ระดับอำเภอไปแจ้งความเองได้ไหม" ยากที่สุดสำหรับทุกวิธี เพราะเอกสารเขียนว่า "ไม่มีอำนาจในการร้องทุกข์กล่าวโทษ" ไม่มีคำว่าแจ้งความเลย
- ข้อตัวหลอก เช่น อายุความ ข้อความกฎหมายเรื่อง "อายุความ 10 ปี" มีอยู่ทั้งในส่วน กข.คจ. และส่วนกองทุนพัฒนาบทบาทสตรี บางครั้งระบบจึงดึงส่วนที่ผิดขึ้นมาก่อน ตรงนี้การกรองด้วย metadata หรือการให้ agent ระบุชื่อกองทุนในคำค้นจะช่วยได้
- ข้อที่ถามด้วยคำว่า "คอนเทนเนอร์" แบบ sparse หาไม่เจอเลย เพราะเอกสารเขียนว่า Docker

ข้อสังเกตเหล่านี้คือสิ่งที่ตัวเลขเฉลี่ยบอกไม่ได้ จึงต้องอ่านรายข้อทุกครั้ง

### วัดคำตอบบนเอกสารของกรมฯ
ใช้ token F1 จากข้อ 8 กับคำถามทุกข้อในชุดทดสอบ รวมถึงข้อที่ไม่มีคำตอบ (ต้องมี LLM ตัวกลาง)

In [ ]:
records = []
for r in testset:
    try:
        pred = answer(r["question"])["answer"]
    except Exception as e:
        print("เรียก LLM ไม่ได้:", type(e).__name__, "— ตรวจ LLM_BASE_URL / LLM_MODEL ใน .env")
        break
    p, rc, f1 = token_prf(pred, r["answer"])
    records.append({"id": r["id"], "ประเภท": r["type"], "P": round(p, 2), "R": round(rc, 2), "F1": round(f1, 2),
                    "คำตอบของระบบ": pred.replace("\n", " ")[:100], "คำตอบเต็ม": pred})

cdd_qa = pd.DataFrame(records)
if len(cdd_qa):
    print(cdd_qa.groupby("ประเภท")[["P", "R", "F1"]].mean().round(2))
    none_rows = cdd_qa[cdd_qa["ประเภท"] == "ไม่มีคำตอบ"]
    print(f"\nข้อที่ไม่มีคำตอบ ตอบว่าไม่พบ {none_rows['คำตอบของระบบ'].str.contains('ไม่พบ').sum()} จาก {len(none_rows)} ข้อ")
cdd_qa.drop(columns="คำตอบเต็ม") if len(cdd_qa) else cdd_qa

ลองดูข้อที่ F1 ต่ำ เช่น ถามว่าสัญญาต้องทำกี่ชุด ระบบตอบว่า "3 ชุด" ซึ่งถูก แต่ F1 ต่ำมาก เพราะคำตอบอ้างอิงเขียนรายละเอียดไว้ยาวกว่า นี่คือข้อจำกัดของ token F1 ที่พูดไว้ในข้อ 8

อีกวิธีหนึ่งคือให้ LLM เป็นผู้ตรวจ (LLM-as-judge) โดยให้ดูคำถาม คำตอบอ้างอิง และคำตอบของระบบ แล้วตัดสินว่าถูกหรือผิด วิธีนี้เข้าใจความหมายได้ดีกว่า แต่ช้ากว่า มีค่าใช้จ่าย และผู้ตรวจเองก็ตัดสินผิดได้ จึงควรสุ่มตรวจผลของผู้ตรวจด้วยคนเสมอ

In [ ]:
from cdd_rag.rag import chat

JUDGE_PROMPT = """คุณเป็นผู้ตรวจคำตอบ ดูคำถาม คำตอบอ้างอิง และคำตอบของระบบ
ตัดสินว่าคำตอบของระบบ "ถูก" หรือ "ผิด" โดยดูที่ความหมาย ไม่ต้องดูว่าใช้คำตรงกันหรือไม่
ถ้าคำตอบอ้างอิงคือ "ไม่พบข้อมูลในเอกสาร" ระบบต้องบอกว่าไม่พบ จึงจะถือว่าถูก
ตอบบรรทัดแรกเป็นคำเดียว: ถูก หรือ ผิด แล้วบรรทัดที่สองอธิบายสั้น ๆ"""

def judge(question, reference, prediction):
    reply = chat([
        {"role": "system", "content": JUDGE_PROMPT},
        {"role": "user", "content": f"คำถาม: {question}\nคำตอบอ้างอิง: {reference}\nคำตอบของระบบ: {prediction}"},
    ])
    first, _, why = reply.strip().partition("\n")
    return first.strip().startswith("ถูก"), why.strip()

if len(cdd_qa):
    by_id = {r["id"]: r for r in testset}
    verdicts = [judge(by_id[i]["question"], by_id[i]["answer"], a) for i, a in zip(cdd_qa["id"], cdd_qa["คำตอบเต็ม"])]
    cdd_qa["ผู้ตรวจ LLM"] = ["ถูก" if ok else "ผิด" for ok, _ in verdicts]
    cdd_qa["เหตุผล"] = [why[:80] for _, why in verdicts]
    print("token F1 เฉลี่ย", round(cdd_qa.F1.mean(), 2), "| ผู้ตรวจ LLM ให้ถูก", (cdd_qa["ผู้ตรวจ LLM"] == "ถูก").mean().round(2))
    print(cdd_qa.groupby("ประเภท").apply(lambda d: pd.Series({"F1": d.F1.mean().round(2), "ถูก (LLM)": (d["ผู้ตรวจ LLM"] == "ถูก").mean().round(2)})))
cdd_qa[["id", "ประเภท", "F1", "ผู้ตรวจ LLM", "คำตอบของระบบ", "เหตุผล"]] if len(cdd_qa) else cdd_qa

ตอนเตรียม lab เจอเรื่องที่น่าสนใจในข้อ q26 เดิมเราตั้งคำถามว่า "อธิบดีคนปัจจุบันชื่ออะไร" ไว้เป็นข้อที่ไม่มีคำตอบ แต่ระบบตอบชื่อออกมาพร้อมอ้างอิง พอตรวจดูก็พบว่าชื่อนี้อยู่ในหน้าคณะที่ปรึกษาของคู่มือจริง คนตั้งคำถามเป็นฝ่ายผิดเอง

มีบทเรียนสองข้อ ข้อแรก ข้อที่เราคิดว่า "ไม่มีคำตอบ" ก็ต้องตรวจเหมือนกัน ข้อสอง เอกสารบอกได้แค่ว่าใครเป็นอธิบดี ณ วันที่จัดทำเอกสาร ไม่ได้บอกว่าเป็น "คนปัจจุบัน" ระบบที่ดีควรบอกด้วยว่าข้อมูลมาจากเอกสารฉบับไหน ปีไหน ซึ่งทำได้จาก field `issued_date` ที่เราออกแบบไว้ใน Lab 01

กิจกรรม (15 นาที) แต่ละกลุ่มเขียนคำถามเพิ่มจากเอกสารใน `data/raw/` กลุ่มละ 5 ข้อ ให้มีอย่างน้อยหนึ่งข้อแบบอ้อมและหนึ่งข้อแบบไม่มีคำตอบ เพิ่มลงใน `data/golden/cdd_testset.csv` พร้อมเลขหน้าและวลีหลักฐาน แล้วรันข้อ 9 ใหม่ ดูว่าคำถามของกลุ่มทำให้คะแนนเปลี่ยนไปแค่ไหน และระบบพลาดข้อไหน

---

## สรุป
| เรื่อง | จำไว้ข้อเดียว |
|---|---|
| Golden set เล็ก | บอกได้แค่ว่าระบบ "ทำงาน" ไม่ได้บอกว่า "ดี" |
| ตัวชี้วัด | RAG สนใจ Recall@k เป็นหลัก · MRR/nDCG ดูคุณภาพการเรียงอันดับ |
| Precision / Recall / F1 | ส่ง chunk มากขึ้น recall ขึ้นแต่ precision ลง ใช้เลือก top_k |
| วัดคำตอบ | token F1 เทียบกับคำตอบที่ถูก ใช้เทียบก่อนและหลังปรับ แต่ต้องอ่านรายข้อประกอบ |
| BM25 vs Dense | ภาษาไทย BM25 ขึ้นกับการตัดคำ · Dense ทนคำพ้อง/พิมพ์ผิดได้ดีกว่า |
| Hybrid | ไม่ได้ดีกว่าเสมอ — น้ำหนักสำคัญ และต้องวัดบนข้อมูลของเราเอง |
| ปรับอะไร | วัดก่อน/หลังเสมอ (Lab 01: ใส่ชื่อเอกสารทุก chunk ทำคะแนนตก) |